In [1]:
import re
import pandas as pd

In [4]:
log_pattern = re.compile(
    r'^(?P<Date>\d{6})\s+(?P<Time>\d{6})\s+(?P<Pid>\d+)\s+'
    r'(?P<Level>[A-Z]+)\s+(?P<Component>[^:]+):\s+(?P<Content>.*)$'
)

parsed_rows = []
malformed = []

with open("HDFS.log", "r", errors="replace") as f:
    for i, line in enumerate(f):
        match = log_pattern.match(line.strip())
        if match:
            parsed_rows.append(match.groupdict())
        else:
            malformed.append((i, line))

df = pd.DataFrame(parsed_rows)
print(f"Parsed: {len(df)}, Malformed: {len(malformed)}")

Parsed: 11175629, Malformed: 0


In [5]:
block_id_pattern = re.compile(r'(blk_-?\d+)')

df.head()

,Date,Time,Pid,Level,Component,Content
0,081109,203518,143,INFO,dfs.DataNode$DataXceiver,Receiving block blk_-1608999687919862906 src: ...
1,081109,203518,35,INFO,dfs.FSNamesystem,BLOCK* NameSystem.allocateBlock: /mnt/hadoop/m...
2,081109,203519,143,INFO,dfs.DataNode$DataXceiver,Receiving block blk_-1608999687919862906 src: ...
3,081109,203519,145,INFO,dfs.DataNode$DataXceiver,Receiving block blk_-1608999687919862906 src: ...
4,081109,203519,145,INFO,dfs.DataNode$PacketResponder,PacketResponder 1 for block blk_-1608999687919...


In [6]:
def extract_block_id(content):
    match_result = block_id_pattern.search(content)
    if match_result:
        return match_result.group(1)
    else :
        return None

df['BlockId'] = df['Content'].apply(extract_block_id)

In [7]:
df.head()

,Date,Time,Pid,Level,Component,Content,BlockId
0,081109,203518,143,INFO,dfs.DataNode$DataXceiver,Receiving block blk_-1608999687919862906 src: ...,blk_-1608999687919862906
1,081109,203518,35,INFO,dfs.FSNamesystem,BLOCK* NameSystem.allocateBlock: /mnt/hadoop/m...,blk_-1608999687919862906
2,081109,203519,143,INFO,dfs.DataNode$DataXceiver,Receiving block blk_-1608999687919862906 src: ...,blk_-1608999687919862906
3,081109,203519,145,INFO,dfs.DataNode$DataXceiver,Receiving block blk_-1608999687919862906 src: ...,blk_-1608999687919862906
4,081109,203519,145,INFO,dfs.DataNode$PacketResponder,PacketResponder 1 for block blk_-1608999687919...,blk_-1608999687919862906


In [8]:
missing_count = df['BlockId'].isnull().sum()
missing_count

np.int64(0)

In [9]:
print(df[['Content', "BlockId"]].head(10))

                                             Content                   BlockId
0  Receiving block blk_-1608999687919862906 src: ...  blk_-1608999687919862906
1  BLOCK* NameSystem.allocateBlock: /mnt/hadoop/m...  blk_-1608999687919862906
2  Receiving block blk_-1608999687919862906 src: ...  blk_-1608999687919862906
3  Receiving block blk_-1608999687919862906 src: ...  blk_-1608999687919862906
4  PacketResponder 1 for block blk_-1608999687919...  blk_-1608999687919862906
5  PacketResponder 2 for block blk_-1608999687919...  blk_-1608999687919862906
6  Received block blk_-1608999687919862906 of siz...  blk_-1608999687919862906
7  Received block blk_-1608999687919862906 of siz...  blk_-1608999687919862906
8  PacketResponder 0 for block blk_-1608999687919...  blk_-1608999687919862906
9  Received block blk_-1608999687919862906 of siz...  blk_-1608999687919862906


In [15]:
anamoly_labels = pd.read_csv("./preprocessed/anomaly_label.csv")
anamoly_labels.head()

,BlockId,Label
0,blk_-1608999687919862906,Normal
1,blk_7503483334202473044,Normal
2,blk_-3544583377289625738,Anomaly
3,blk_-9073992586687739851,Normal
4,blk_7854771516489510256,Normal


In [16]:
df_labeled = df.merge(anamoly_labels, on="BlockId", how="left")
df_labeled.head(10)

,Date,Time,Pid,Level,Component,Content,BlockId,Label
0,081109,203518,143,INFO,dfs.DataNode$DataXceiver,Receiving block blk_-1608999687919862906 src: ...,blk_-1608999687919862906,Normal
1,081109,203518,35,INFO,dfs.FSNamesystem,BLOCK* NameSystem.allocateBlock: /mnt/hadoop/m...,blk_-1608999687919862906,Normal
2,081109,203519,143,INFO,dfs.DataNode$DataXceiver,Receiving block blk_-1608999687919862906 src: ...,blk_-1608999687919862906,Normal
3,081109,203519,145,INFO,dfs.DataNode$DataXceiver,Receiving block blk_-1608999687919862906 src: ...,blk_-1608999687919862906,Normal
4,081109,203519,145,INFO,dfs.DataNode$PacketResponder,PacketResponder 1 for block blk_-1608999687919...,blk_-1608999687919862906,Normal
5,081109,203519,145,INFO,dfs.DataNode$PacketResponder,PacketResponder 2 for block blk_-1608999687919...,blk_-1608999687919862906,Normal
6,081109,203519,145,INFO,dfs.DataNode$PacketResponder,Received block blk_-1608999687919862906 of siz...,blk_-1608999687919862906,Normal
7,081109,203519,145,INFO,dfs.DataNode$PacketResponder,Received block blk_-1608999687919862906 of siz...,blk_-1608999687919862906,Normal
8,081109,203519,147,INFO,dfs.DataNode$PacketResponder,PacketResponder 0 for block blk_-1608999687919...,blk_-1608999687919862906,Normal
9,081109,203519,147,INFO,dfs.DataNode$PacketResponder,Received block blk_-1608999687919862906 of siz...,blk_-1608999687919862906,Normal


In [ ]:
missing_labels_after_merge = df_labeled['Label'].isnull().sum()
print(missing_labels_after_merge)
print(df_labeled[['BlockId', 'Label']].head(10))


0
                           BlockId   Label
0         blk_-1608999687919862906  Normal
1         blk_-1608999687919862906  Normal
2         blk_-1608999687919862906  Normal
3         blk_-1608999687919862906  Normal
4         blk_-1608999687919862906  Normal
...                            ...     ...
11175624  blk_-6171368032583208892  Normal
11175625   blk_6195025276114316035  Normal
11175626  blk_-3339773404714332088  Normal
11175627   blk_1037231945509285002  Normal
11175628   blk_4258862871822415442  Normal

[11175629 rows x 2 columns]


In [24]:
events_per_block = df_labeled.groupby("BlockId").size()
events_per_block = events_per_block.reset_index(name="EventCount")

labeled_events = events_per_block.merge(anamoly_labels, on="BlockId", how="left")
print(labeled_events.head(10))

                    BlockId  EventCount   Label
0  blk_-1000002529962039464          13  Normal
1   blk_-100000266894974466          28  Normal
2  blk_-1000007292892887521          13  Normal
3  blk_-1000014584150379967          29  Normal
4  blk_-1000028658773048709          19  Normal
5   blk_-100004553717737248          13  Normal
6  blk_-1000054577281647820          30  Normal
7  blk_-1000057191487947536          19  Normal
8  blk_-1000083860370843431          23  Normal
9  blk_-1000095285706020638          24  Normal


In [25]:
summary = labeled_events.groupby("Label")["EventCount"].describe()
print(summary)

            count       mean        std   min   25%   50%   75%    max
Label                                                                 
Anomaly   16838.0  17.119017  12.409644   2.0   4.0  20.0  26.0  284.0
Normal   558223.0  19.503637   4.775583  13.0  19.0  19.0  20.0  298.0
